In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cmocean
import os
from scipy.interpolate import griddata



filepath = r"C:\Users\marqjace\seaglider\sg266\sg266_2024_10_21_TH_Line_timeseries.nc"
figures_folder = r"C:\Users\marqjace\seaglider\sg266\figures"
os.makedirs(figures_folder, exist_ok=True)  # exist_ok=True prevents errors if the folder already exists

# Open the dataset
ds = xr.open_dataset(filepath)

# Variables
wl700nm = ds.wlbbfl2_sig700nm
time_coverage_start = ds.attrs['time_coverage_start']
time_coverage_end = ds.attrs['time_coverage_end']

# Convert time variables to the same format
ctd_time = pd.to_datetime(ds.ctd_time.values, unit='s', origin='unix')
wl_time_dt = pd.to_datetime(ds.wlbbfl2_time.values, unit='s', origin='unix')

# Interpolate ctd_depth onto aa4831_time
wl_depth = np.interp(wl_time_dt.astype(np.int64), ctd_time.astype(np.int64), ds.ctd_depth)

# Calculate and print the number of days the mission lasted
difference = wl_time_dt.max() - wl_time_dt.min()
num_days = difference.days
print(f'The mission lasted {num_days} days.') 

wl_time_timestamps = wl_time_dt.astype(np.int64) // 10**9

# Time vs Depth Grid (using the ctd_data_point dimension)
xn2, yn2 = int(num_days * 4), 120 # (The number of mission days multiplied by 4 dives per day (on average), 1000m / 5m per dive = 120 points)
xmin2, xmax2 = wl_time_timestamps.min(), wl_time_timestamps.max()
ymin2, ymax2 = 0, 600
xgrid2 = np.linspace(xmin2, xmax2, xn2)
ygrid2 = np.linspace(ymin2, ymax2, yn2)
Xgrid2, Ygrid2 = np.meshgrid(xgrid2, ygrid2)

wl_interp = griddata((wl_time_timestamps, wl_depth), wl700nm, (Xgrid2, Ygrid2), method='linear')

sci_variables = {
    'wlbbfl2_sig700nm': wl700nm
}

for var_name, var in sci_variables.items():
    print(f'Processing {var_name} data....')

    if var_name == 'wlbbfl2_sig700nm':
        units = f'$m^-1$'

    var_directory = figures_folder + f'\{var_name}'
    os.makedirs(var_directory, exist_ok=True)  # exist_ok=True prevents errors if the folder already exists

    # Raw Scatter Plot
    plt.figure(figsize=(10, 5), dpi=300)
    plt.scatter(wl_time_dt, wl_depth, c=var, cmap=cmocean.cm.algae)
    plt.colorbar(label=f'{units}')
    plt.clim(0,300)
    plt.gca().invert_yaxis()
    plt.title(f'{time_coverage_start} - {time_coverage_end}')
    plt.xlabel('Time')
    plt.ylabel('Depth (m)')
    plt.ylim(600,0)
    plt.grid(alpha=0.5)
    plt.savefig(f'{var_directory}/raw_{var_name}_mission.png')
    plt.close()
    print(f'raw_{var_name}_mission.png created')

    # Gridded Scatter Plot
    plt.figure(figsize=(10, 5), dpi=300)
    plt.scatter(Xgrid2, Ygrid2, c=wl_interp, cmap=cmocean.cm.algae)
    plt.colorbar(label=f'{units}')
    plt.gca().invert_yaxis()
    plt.title(f'{time_coverage_start} - {time_coverage_end}')
    plt.xlabel('Time')
    plt.ylabel('Depth (m)')
    plt.clim(0,300)
    plt.ylim(600,0)
    plt.grid(alpha=0.5)
    plt.savefig(f'{var_directory}/gridded_{var_name}_mission.png')
    plt.close()
    print(f'gridded_{var_name}_mission.png created')

    # Contour Plot
    levels = np.arange(0, 300, 50)
    plt.figure(figsize=(10, 5), dpi=300)
    contour = plt.contourf(Xgrid2, Ygrid2, wl_interp, levels=levels, cmap=cmocean.cm.algae)
    contour_lines = plt.contour(Xgrid2, Ygrid2, wl_interp, levels=levels, colors='black', linewidths=0.5)
    plt.clabel(contour_lines, inline=True, fontsize=8, fmt='%1.1f')
    plt.colorbar(contour, label=f'{units}')
    plt.gca().invert_yaxis()
    plt.title(f'{time_coverage_start} - {time_coverage_end}')
    plt.xlabel('Time')
    plt.ylabel('Depth (m)')
    plt.ylim(600,0)
    plt.grid(alpha=0.5)
    plt.savefig(f'{var_directory}/contour_{var_name}_mission.png')
    plt.close()
    print(f'contour_{var_name}_mission.png created')

print('Done!')

<xarray.Dataset> Size: 87MB
Dimensions:                                   (gps_info: 651,
                                               wlbbfl2_data_point: 146875,
                                               legato_data_point: 325486,
                                               depth_data_point: 399378,
                                               aa4831_data_point: 324152,
                                               sg_data_point: 458954,
                                               ctd_data_point: 325486,
                                               trajectory: 217, dive: 217)
Coordinates:
    ctd_time                                  (ctd_data_point) datetime64[ns] 3MB ...
    ctd_depth                                 (ctd_data_point) float32 1MB ...
    latitude                                  (ctd_data_point) float32 1MB ...
    longitude                                 (ctd_data_point) float32 1MB ...
  * trajectory                                (trajectory) int32 868B 1 ... 217
Dimensions without coordinates: gps_info, wlbbfl2_data_point,
                                legato_data_point, depth_data_point,
                                aa4831_data_point, sg_data_point,
                                ctd_data_point, dive
Data variables: (12/86)
    gps_info_dive_number                      (gps_info) int32 3kB ...
    wlbbfl2_data_point_dive_number            (wlbbfl2_data_point) int32 588kB ...
    legato_data_point_dive_number             (legato_data_point) int32 1MB ...
    depth_data_point_dive_number              (depth_data_point) int32 2MB ...
    aa4831_data_point_dive_number             (aa4831_data_point) int32 1MB ...
    sg_data_point_dive_number                 (sg_data_point) int32 2MB ...
    ...                                        ...
    end_longitude                             (dive) float32 868B ...
    depth_avg_curr_east                       (dive) float32 868B ...
    depth_avg_curr_north                      (dive) float32 868B ...
    depth_avg_curr_qc                         (dive) |S1 217B ...
    latlong_qc                                (dive) |S1 217B ...
    glider                                    |S12 12B ...
Attributes: (12/47)
    project:                         2024_10_21_TH_Line
    title:                           Physical, chemical, and biological data ...
    summary:                         SG266 2024_10_21_TH_Line
    source:                          Seaglider SG266
    references:                      http://data.nodc.noaa.gov/accession/0092291
    processing_level:                1.12
    ...                              ...
    date_modified:                   2024-12-04T20:12:28Z
    uuid:                            f8146e2a-b27a-11ef-91ac-91df858c20f4
    base_station_version:            3.0.2
    base_station_micro_version:      0
    quality_control_version:         1.12
    Conventions:                     CF-1.6